# pplx-embed-v2 context 0.6B — contextual chunk embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/pplx-embed-v2-context-0.6b-84b1/colab/pplx-embed-v2-context-0.6b-contextual-rag.ipynb)

[`perplexity-ai/pplx-embed-v2-context-0.6b`](https://huggingface.co/perplexity-ai/pplx-embed-v2-context-0.6b) is Perplexity's **0.6B multimodal contextual embedding** model (text, images, visual documents). It embeds each **document chunk with the surrounding chunks in view**, then pools one **1024-dimensional** vector per chunk. That is the difference from the [late-interaction sibling](pplx-embed-v2-late-0.6b-page-search.ipynb) in this repo, which emits one 128-d vector per token and scores a page with MaxSim. Here the index is ordinary dense vectors: one row per non-overlapping chunk, ready for cosine / ANN.

It shares an embedding space with the **1024-dim MRL prefix** of [`pplx-embed-v2-context-9b`](https://huggingface.co/perplexity-ai/pplx-embed-v2-context-9b), so you can embed documents with one size and query with the other. This notebook stays on 0.6B so it fits a free T4.

**Why this model.** A chunk like *“It comes with a two-year warranty”* does not name the product. Contextual mode lets the warranty sentence see “Cedar laptop” in the previous chunk; isolated (non-contextual) mode cannot. Demo 1 is that contrast, taken from [tomaarsen’s announcement](https://x.com/tomaarsen/status/2117618555210641590).

**API (from the card).** Queries go through `encode_queries`. Chunks go through `encode`. Encoding a query with `encode` silently degrades retrieval. Chunks in contextual mode must be **non-overlapping**. Default outputs are **unnormalized int8** (float32 integers in `[-128, 127]`); compare with cosine, or pass `normalize_embeddings=True` and use the dot product. This notebook normalizes. Load with `trust_remote_code=True`. Needs `transformers>=5.4.0`.

**Licence:** the Hub card lists **no license tag** (the late 0.6B sibling is MIT). Review the card before any use beyond this demo.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has ~15 GB of VRAM, **no native bfloat16**, and **no FlashAttention 2**. The Hub file is ~2.4 GB of float32; this notebook loads **float16 + SDPA**. Weights are about **1.2 GB** in float16; a short text encode should stay around 2–4 GB. The card’s image snippet uses `dtype=torch.bfloat16`; T4 uses float16 instead. Image encodes keep `max_pixels` modest so the vision pass fits.

**Fast path vs slow path.** The card asks for `flash-linear-attention` and `causal-conv1d` (Qwen3.5 hybrid linear-attention). **Colab default is the slow path:** those packages are not installed. Building them on a free T4 often OOMs the ~12.7 GB host RAM (or fails because `causal-conv1d` wants `nvcc`). Transformers uses a slower PyTorch fallback instead. This notebook encodes **one document (or query) per forward pass** (`ENCODE_BATCH=1`) so packed-batch linear-attention state cannot leak across examples ([transformers#49472](https://github.com/huggingface/transformers/issues/49472)). Set `INSTALL_FAST_PATH = True` in the install cell only if you have `nvcc` and enough RAM.

**Links:** [model card](https://huggingface.co/perplexity-ai/pplx-embed-v2-context-0.6b) · [paper](https://arxiv.org/abs/2610.38854) · [late-interaction notebook](pplx-embed-v2-late-0.6b-page-search.ipynb) · [announcement](https://x.com/tomaarsen/status/2117618555210641590)


In [ ]:
%pip install -q "transformers==5.20.1" pillow torchvision tqdm pypdfium2==5.14.0

# Card floor is transformers>=5.4.0 (custom PplxContextualModel on Qwen3.5).
# 5.20.1 matches the late-interaction sibling and other T4 notebooks here.
# Do not pip-install torch: Colab already has a CUDA wheel.
# Use %pip, not `python -I -m pip` (isolated mode drops Colab site-packages).
# pillow / torchvision: Qwen3VLProcessor imports them for encode_images.
# pypdfium2 renders one public arXiv page in demo 3.
# Do not install flash-attn (T4 cannot run FlashAttention 2).
#
# Free-T4 default: do not install flash-linear-attention or causal-conv1d.
# Building them on ~12.7 GB host RAM often kills the Colab VM (or fails
# because causal-conv1d wants nvcc). Transformers already has a slow
# PyTorch fallback; ENCODE_BATCH=1 handles packed-batch state leak
# (transformers#49472). Flip this only if you have nvcc and enough RAM.
INSTALL_FAST_PATH = False

if INSTALL_FAST_PATH:
    import subprocess
    import sys

    fast = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "flash-linear-attention", "causal-conv1d"],
        capture_output=True,
        text=True,
    )
    if fast.returncode == 0:
        print("fast path: flash-linear-attention + causal-conv1d installed")
    else:
        print("fast path packages failed to install; using the slow torch fallback")
        err = (fast.stderr or fast.stdout or "").strip().splitlines()
        for line in err[-8:]:
            print(" ", line)
else:
    print("slow path (Colab default): skipped flash-linear-attention + causal-conv1d")


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


## 1. Check the GPU

Print `torch.cuda.is_available()`, the GPU name, and free VRAM. T4 (`sm_75`) has no native bfloat16 — **float16** is the path this notebook uses.


In [ ]:
import gc

import torch
import transformers
from packaging.version import Version

print("cuda available:", torch.cuda.is_available())
assert Version(transformers.__version__) >= Version("5.4.0"), (
    f"transformers {transformers.__version__} is loaded; pplx-embed-v2-context needs 5.4.0+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
DEVICE = "cuda"
# Card image snippet uses bfloat16. T4 has no native bf16.
DTYPE = torch.float16 if major < 8 else torch.bfloat16
MODEL_ID = "perplexity-ai/pplx-embed-v2-context-0.6b"

free_b, total_b = torch.cuda.mem_get_info()
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"dtype: {DTYPE}")
print(f"VRAM total: {total_b / 1024**3:.2f} GB | free: {free_b / 1024**3:.2f} GB")
if major < 8:
    print("No native bfloat16 on this GPU — staying on float16.")


def print_vram(prefix=""):
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"{prefix}VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | "
        f"peak: {peak:.2f} GB | free: {free_now / 1024**3:.2f}/{total_now / 1024**3:.2f} GB"
    )


## 2. Load the contextual encoder

`AutoModel.from_pretrained(..., trust_remote_code=True)` is the card’s loader. The custom code (`modeling_pplx_contextual.py`) adds `encode` / `encode_queries` / `encode_images`. Those methods read `model.tokenizer`; transformers 5.20 does not attach one on `AutoModel.from_pretrained`, so this cell loads `AutoTokenizer` from the same repo and sets `model.tokenizer`. SDPA first; eager if SDPA refuses the hybrid stack.


In [ ]:
from transformers import AutoModel, AutoTokenizer

try:
    from transformers.utils import (
        is_causal_conv1d_available,
        is_flash_linear_attention_available,
    )

    FAST_PATH = bool(
        is_flash_linear_attention_available() and is_causal_conv1d_available()
    )
except Exception:
    try:
        import causal_conv1d  # noqa: F401
        import fla  # noqa: F401

        FAST_PATH = True
    except ImportError:
        FAST_PATH = False

# Slow-path fallbacks can leak linear-attention state across packed rows
# (transformers#49472). One example per forward pass avoids that.
ENCODE_BATCH = 32 if FAST_PATH else 1
print("linear-attention fast path:", FAST_PATH, "| encode batch_size:", ENCODE_BATCH)


def load_model(attn):
    return (
        AutoModel.from_pretrained(
            MODEL_ID,
            trust_remote_code=True,
            dtype=DTYPE,
            attn_implementation=attn,
        )
        .to(DEVICE)
        .eval()
    )


try:
    model = load_model("sdpa")
    print("attention: sdpa")
except Exception as exc:
    print("sdpa load failed, retrying eager:", exc)
    model = load_model("eager")
    print("attention: eager")

# encode / encode_queries / encode_images read model.tokenizer.
# AutoModel.from_pretrained does not attach one on transformers 5.20.
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side="right",
)
model.tokenizer = tokenizer
print("tokenizer:", type(tokenizer).__name__, "vocab:", len(tokenizer))

print("embedding_dim:", getattr(model.config, "embedding_dim", None))
print("query_prefix:", repr(getattr(model.config, "query_prefix", None)))
print("document_prefix:", repr(getattr(model.config, "document_prefix", None)))
print_vram("after load  ")


def encode_docs(documents, **kwargs):
    return model.encode(
        documents,
        batch_size=ENCODE_BATCH,
        normalize_embeddings=True,
        **kwargs,
    )


def encode_q(queries, **kwargs):
    return model.encode_queries(
        queries,
        batch_size=ENCODE_BATCH,
        normalize_embeddings=True,
        **kwargs,
    )


## 3. Contextual vs isolated retrieval

Two products share the **same** warranty sentence. Only the *previous* chunk names the product. The query is *“What is the warranty on the Cedar laptop?”*

- **Contextual** `encode(documents)` — each document’s chunks see each other. The Cedar warranty chunk should outscore the identical Maple sentence.
- **Non-contextual** `encode([[chunk], ...])` — the card’s isolated mode. The two warranty strings are the same, so their scores should match.

Chunks inside each document do not overlap.


In [ ]:
import numpy as np

DOCS = [
    [
        "The Cedar laptop is a 14-inch ultrabook for frequent travelers.",
        "It comes with a two-year warranty.",
        "The chassis is recycled aluminum and the charger is 65W.",
    ],
    [
        "The Maple tablet is an 11-inch Android slate for classrooms.",
        "It comes with a two-year warranty.",
        "A folio case and a stylus ship in the box.",
    ],
    [
        "The Birch headphones fold into a carry pouch.",
        "Drivers are 40 mm and the cable is detachable.",
        "Battery life is rated at 30 hours.",
    ],
]
QUERY = "What is the warranty on the Cedar laptop?"

# One embedding per chunk, with sibling chunks in view.
ctx_rows = encode_docs(DOCS)
# Card non-contextual mode: flatten to singleton documents.
flat = [[chunk] for doc in DOCS for chunk in doc]
iso_rows = encode_docs(flat)
q = encode_q([[QUERY]])[0][0]

records = []
flat_i = 0
for doc_i, doc in enumerate(DOCS):
    ctx = ctx_rows[doc_i]
    for chunk_i, text in enumerate(doc):
        ctx_score = float(ctx[chunk_i] @ q)
        iso_score = float(iso_rows[flat_i][0] @ q)
        records.append((doc_i, chunk_i, text, ctx_score, iso_score))
        flat_i += 1

print(f"Q: {QUERY}\n")
print(f"{'doc':>3} {'ch':>3}  {'ctx':>7}  {'iso':>7}  chunk")
print("-" * 88)
for doc_i, chunk_i, text, ctx_score, iso_score in records:
    print(f"{doc_i:3d} {chunk_i:3d}  {ctx_score:7.4f}  {iso_score:7.4f}  {text}")

cedar_w = next(r for r in records if r[0] == 0 and "warranty" in r[2])
maple_w = next(r for r in records if r[0] == 1 and "warranty" in r[2])
print()
print(f"Cedar warranty  ctx={cedar_w[3]:.4f}  iso={cedar_w[4]:.4f}")
print(f"Maple warranty  ctx={maple_w[3]:.4f}  iso={maple_w[4]:.4f}")
print(f"ctx gap (Cedar - Maple): {cedar_w[3] - maple_w[3]:+.4f}")
print(f"iso gap (Cedar - Maple): {cedar_w[4] - maple_w[4]:+.4f}")
print(
    "Isolated scores match on the shared sentence; contextual mode "
    "should prefer Cedar because the product name lives in another chunk."
)
print_vram("demo 1  ")


## 4. Mixed-language chunk search

The same 1024-d space is multilingual. An English query should still rank a Spanish Cedar-warranty chunk above German / Japanese distractors. A French query should hit the Aspen refund sentence.


In [ ]:
import numpy as np

MIXED = [
    ["La garantía del portátil Cedar cubre piezas y mano de obra durante dos años."],
    ["Die Maple-Tablethülle liegt dem Karton bei; eine Garantie ist nicht enthalten."],
    ["Birchヘッドホンのバッテリーは30時間持続します。"],
    ["The Cedar laptop chassis is recycled aluminum."],
    ["Le remboursement du moniteur Aspen est de 14 jours."],
]
MIXED_QUERIES = [
    "How long is the Cedar laptop warranty?",
    "Quelle est la durée de remboursement du moniteur Aspen ?",
]

mixed_rows = encode_docs(MIXED)
q_rows = encode_q([[q] for q in MIXED_QUERIES])

for q_text, q_row in zip(MIXED_QUERIES, q_rows):
    scores = np.array([float(row[0] @ q_row[0]) for row in mixed_rows])
    order = np.argsort(-scores)
    print(f"Q: {q_text}")
    for rank, idx in enumerate(order, start=1):
        mark = "  << best" if rank == 1 else ""
        print(f"  {rank}. {scores[idx]:.4f}  {MIXED[idx][0]}{mark}")
    print()
print_vram("demo 2  ")


## 5. Visual-document chunk retrieval

`encode_images` splits a page into horizontal bands and returns one 1024-d vector per band (same space as text). Query with `encode_queries`. Page score is the **max** band score.

The notebook downloads two public images itself: the first page of *Attention Is All You Need* (arXiv PDF) and Hugging Face’s cat photo. `chunks=4` and a capped pixel budget keep the vision pass on a T4.


In [ ]:
import io

import matplotlib.pyplot as plt
import numpy as np
import requests
from PIL import Image

UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}


def first_page(url, longest=1024):
    import pypdfium2 as pdfium

    response = requests.get(url, headers=UA, timeout=120)
    response.raise_for_status()
    pdf = pdfium.PdfDocument(response.content)
    image = pdf[0].render(scale=120 / 72).to_pil().convert("RGB")
    image.thumbnail((longest, longest))
    return image


def fetch_image(url, longest=1024):
    response = requests.get(url, headers=UA, timeout=120)
    response.raise_for_status()
    image = Image.open(io.BytesIO(response.content)).convert("RGB")
    image.thumbnail((longest, longest))
    return image


paper = first_page("https://arxiv.org/pdf/1706.03762")
cat = fetch_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
)
pages = [
    ("Attention Is All You Need (page 1)", paper),
    ("HF cat photo", cat),
]
for title, image in pages:
    print(f"{title}: {image.size}")

# Card default max_pixels=1_800_000 is large for a T4 vision pass.
image_rows = model.encode_images(
    [image for _title, image in pages],
    chunks=4,
    min_pixels=16_384,
    max_pixels=400_000,
    batch_size=1,
    normalize_embeddings=True,
)
for title, row in zip((t for t, _ in pages), image_rows):
    print(f"{title}: bands x dim = {tuple(row.shape)}")

IMG_QUERIES = [
    "scaled dot-product attention and multi-head attention",
    "a fluffy cat standing in grass",
]
img_q = encode_q([[q] for q in IMG_QUERIES])

fig, axes = plt.subplots(len(IMG_QUERIES), len(pages), figsize=(10, 8))
for row_i, (q_text, q_row) in enumerate(zip(IMG_QUERIES, img_q)):
    print(f"\nQ: {q_text}")
    page_scores = []
    for col, ((title, _image), bands) in enumerate(zip(pages, image_rows)):
        band_scores = bands @ q_row[0]
        best = float(np.max(band_scores))
        page_scores.append(best)
        print(f"  {title}: max={best:.4f}  bands={np.round(band_scores, 4)}")
    winner = int(np.argmax(page_scores))
    for col, (title, image) in enumerate(pages):
        ax = axes[row_i, col]
        ax.imshow(image)
        ax.set_axis_off()
        mark = "  << best" if col == winner else ""
        ax.set_title(f"{page_scores[col]:.3f}{mark}\n{title}", fontsize=8)
fig.suptitle("Text query → image bands (max chunk score)", fontsize=13)
fig.tight_layout()
plt.show()
print_vram("demo 3  ")


## 6. Caveats and cleanup

**What this notebook is for.** A free-T4 demo of *contextual* chunk embeddings: `encode` vs `encode_queries`, contextual vs isolated scoring, a short multilingual search, and `encode_images` band retrieval.

**What it is not.** The [late-interaction page-search notebook](pplx-embed-v2-late-0.6b-page-search.ipynb) (ColBERT-style MaxSim, MIT). The 9B contextual checkpoint. A production ANN service.

**API reminders (card).**
- Contextual chunks must be **non-overlapping**.
- Queries: `encode_queries`. Documents: `encode`. Do not encode queries with `encode`.
- Default vectors are unnormalized int8. This notebook sets `normalize_embeddings=True` and uses the dot product.
- The text API does not truncate. Split documents that exceed the configured length.

**T4 / kernels.** float16 + SDPA. The Colab default is the slow path: `flash-linear-attention` / `causal-conv1d` are not installed, and `ENCODE_BATCH=1` already handles that. Scores on the fallback will not match a kernel-backed A100/H100 run. Do not install `flash-attn`.

**Licence.** The Hub card lists **no license tag**. The late 0.6B sibling is MIT; do not assume the same here.

**`trust_remote_code`.** `modeling_pplx_contextual.py` and `image_processing.py` execute locally. Review them, and pin a revision if you need a frozen snapshot. `encode` reads `model.tokenizer`; this notebook loads `AutoTokenizer` from the same repo and attaches it after `AutoModel.from_pretrained`.

**Related:** [late-interaction notebook](pplx-embed-v2-late-0.6b-page-search.ipynb) · [paper](https://arxiv.org/abs/2610.38854) · [9B contextual](https://huggingface.co/perplexity-ai/pplx-embed-v2-context-9b)

Free the GPU before you open another notebook:


In [ ]:
print_vram("before cleanup  ")
try:
    del model
except NameError:
    pass
try:
    del tokenizer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup  ")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("Contextual encoder released. Re-run the load cell if you want another encode.")
